In [14]:
import torch
import torch.nn as nn

# ── 1. DATA ─────────────────────────────────
torch.manual_seed(0)
categories =['Rings', 'Chain', 'Banlge','Pendant']
n = 300

def make(n, w_min, w_max, l_min, l_max, label):
    # TODO: random weights, random lengths, stack into 2 columns, labels
    w = torch.rand(n)*(w_max -w_min) +w_min
    l = torch.rand(n)*(l_max-l_min)+ l_min
    x= torch.stack([w,l],dim=1)
    y = torch.full((n,), label)
    return x, y

# TODO: call make() 4 times
ring_col, ring_label = make(n,2,8,15,22,0)
chain_col, chain_label = make(n,8,30,400,600,1)
bangle_col, bangle_label = make(n,15,40,55,70,2)
pendant_col, pendant_label = make(n,1,10,20,45,3)
# TODO: join all x with torch.cat, join all y with torch.cat
all_col = torch.cat([ring_col,chain_col,bangle_col,pendant_col])
all_label = torch.cat([ring_label,chain_label,bangle_label,pendant_label])
# ── 2. PREPARE ──────────────────────────────
# TODO: shuffle (randperm)
perm = torch.randperm(len(all_col))
all_col = all_col[perm]
all_label=all_label[perm]
# TODO: split 80/20
split = int(0.8*len(all_col))
training_col,test_col = all_col[:split],all_col[split:]
taining_label, test_label = all_label[:split],all_label[split:]

# TODO: standardise using TRAIN mean and std only
col_mean = training_col.mean(dim=0)
col_std = training_col.std(dim=0)
tain_col_n = (training_col-col_mean)/col_std
test_col_n =   (test_col-col_mean)/col_std
# ── 3. MODEL ────────────────────────────────
class CategoryModel(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # TODO: hidden layer, relu, output layer (how many outputs now?)
        self.hidden = nn.Linear(2,hidden_size)
        self.relu = nn.ReLU()
        self.output= nn.Linear(hidden_size,4)

    def forward(self, x):
        # TODO: pass x through the 3 layers
        x=self.hidden(x)
        x=self.relu(x)
        x=self.output(x)
        return x

# ── 4. TRAIN ────────────────────────────────
# TODO: create model, loss_fn, optimizer
torch.manual_seed(0)
model = CategoryModel(16)
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(),lr=0.01)
# TODO: loop 1000 times: forward → loss → zero_grad → backward → step
for i in range(3000):
    score = model(tain_col_n)
    prediction = loss_fn(score,taining_label)
    optimiser.zero_grad()
    prediction.backward()
    optimiser.step()
    if i % 200 == 0:
        print(f'loss value:{prediction.item():.4f}')
# TODO: print loss every 200 steps

# ── 5. EVALUATE ─────────────────────────────
# TODO: inside no_grad: scores → argmax → compare with y_test → accuracy
with torch.no_grad():
    test_eval = model(test_col_n)
    prdeiction = test_eval.argmax(dim=1)
    result = (prdeiction == test_label)
    wrongValue = (prdeiction != test_label)
    accuracy = result.float().mean()
# TODO: print accuracy
    print('accuracy value:', accuracy*100)
    print ('wrong values ',test_col[wrongValue] )

loss value:1.2853
loss value:0.2924
loss value:0.1301
loss value:0.0815
loss value:0.0651
loss value:0.0576
loss value:0.0536
loss value:0.0511
loss value:0.0495
loss value:0.0485
loss value:0.0477
loss value:0.0472
loss value:0.0468
loss value:0.0465
loss value:0.0463
accuracy value: tensor(97.9167)
wrong values  tensor([[ 5.0974, 20.1383],
        [ 3.3980, 22.1279],
        [ 6.1722, 20.0483],
        [ 3.1898, 21.4790],
        [ 5.4507, 21.3279]])
